# Document Parsing using PaddleOCR-VL/PaddleOCR-VL-1.5 and OpenVINO

![PaddleOCR-VL-1.5 demo image](https://raw.githubusercontent.com/cuicheng01/PaddleX_doc_images/refs/heads/main/images/paddleocr_vl_1_5/PaddleOCR-VL-1.5.png)

PaddleOCR-VL-1.5 is an advanced next-generation model of PaddleOCR-VL, achieving a new state-of-the-art accuracy of 94.5% on Omni Doc Bench v1.5. To rigorously evaluate robustness against real-world physical distortions—including scanning artifacts, skew, warping, screen photography, and illumination—we propose the Real5 Omni Doc Bench benchmark. Experimental results demonstrate that this enhanced model attains SOTA performance on the newly curated benchmark. Furthermore, we extend the model’s capabilities by incorporating seal recognition and text spotting tasks, while remaining a 0.9B ultra-compact VLM with high efficiency.

This notebook is designed to be as self-contained as possible:

- Download the original PaddleOCR-VL pretrained model.
- Replace the downloaded model's `modeling_paddleocr_vl.py` with the local patched version.
- Convert to OpenVINO IR.
- Validate the OpenVINO pipeline.


#### Table of content:

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Compress model](#Compress-model)
- [Run model inference](#Run-model-inference)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is using a custom branch of optimum-intel. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/paddleocr_vl/paddleocr_vl.ipynb" />


## Prerequisites

[back to top ⬆️](#Table-of-content:)

In [3]:
import requests
from pathlib import Path

utility_files = ["cmd_helper.py", "notebook_utils.py", "pip_helper.py"]
base_utility_url = "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/"

print("🔍 检查并下载 OpenVINO 工具脚本...")

for utility in utility_files:
    local_file = Path(utility)
    if local_file.exists():
        print(f"✅ {utility} 已存在，跳过下载。")
    else:
        print(f"📥 正在下载 {utility} ...")
        try:
            r = requests.get(base_utility_url + utility)
            r.raise_for_status()  # 如果 HTTP 请求出错（如 404），会抛出异常
            with open(utility, "w", encoding="utf-8") as f:
                f.write(r.text)
            print(f"✔️  {utility} 下载成功！")
        except requests.exceptions.RequestException as e:
            print(f"❌ 下载 {utility} 失败: {e}")

print("✨ 工具脚本准备完成！")

🔍 检查并下载 OpenVINO 工具脚本...
✅ cmd_helper.py 已存在，跳过下载。
✅ notebook_utils.py 已存在，跳过下载。
📥 正在下载 pip_helper.py ...
✔️  pip_helper.py 下载成功！
✨ 工具脚本准备完成！


In [1]:
from pip_helper import pip_install
import platform

# Upgrade pip
pip_install("-U", "pip")

# Install PyTorch CPU wheels explicitly (avoid CUDA wheels)
# NOTE: This index hosts PyTorch wheels only. Do NOT install unrelated packages here.
pip_install(
    "-U",
    "--index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch==2.8.0",
    "torchvision==0.23.0",
    "torchaudio==2.8.0",
)

# Install the rest dependencies from PyPI (keep existing versions/constraints)
pip_install(
    "-q",
    "-U",
    "openvino>=2025.4.1",
    "numpy>=1.21.6",
    "pillow",
    "opencv-python>=4.11.0.86",
    "modelscope>=1.26.0",
    "huggingface-hub>=0.32.4",
    "tqdm>=4.67.1",
    "requests",
    "fastapi",
    "uvicorn",
    "python-multipart",
    "gradio==4.19",
    "transformers==4.54.0",
    "nncf>=2.18.0",
    "datasets>=3.6.0",
    "protobuf>=6.32.0",
    "sentencepiece>=0.2.1",
    "einops>=0.8.1",
)

# macOS compatibility: keep numpy < 2.0 if required by some packages
if platform.system() == "Darwin":
    pip_install("-U", "numpy<2.0")

print("✅ Dependencies installed.")

# Device selection widget (required by CI checks)
# Note: Must be called after openvino is installed (Cell 3)
from notebook_utils import device_widget

device = device_widget("CPU")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("paddleocr_vl.ipynb")

Looking in indexes: http://mirrors.baidubce.com/pypi/simple/
Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 MB 6.4 MB/s  0:00:29m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 3.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 207.6 kB/s  0:01:59 eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 2.0 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [torch]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [torch]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [torchaudio]5 [torchaudio]]


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

✅ Dependencies installed.


In [2]:
from notebook_utils import device_widget

device = device_widget("CPU")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("paddleocr_vl.ipynb")

In [3]:
import os
import shutil
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PATCH_MODELING_FILE = NOTEBOOK_DIR / "modeling_paddleocr_vl.py"

# Where to download the original pretrained model
CACHE_DIR = NOTEBOOK_DIR / "_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Candidate model IDs (set one explicitly if needed)
# If you know the exact ID, set it here to avoid retries.
MODEL_ID = os.environ.get("PADDLEOCR_VL_PRETRAINED_ID", "PaddlePaddle/PaddleOCR-VL-1.5")

# OpenVINO export output dir
OV_OUT_DIR = NOTEBOOK_DIR / "ov_paddleocr_vl_model"

assert PATCH_MODELING_FILE.exists(), f"Missing patch file: {PATCH_MODELING_FILE}"

In [4]:
def download_pretrained_model(model_id: str, cache_dir: Path) -> Path:
    """Download pretrained model into cache_dir and return the local folder path."""
    # Prefer ModelScope if installed
    try:
        from modelscope import snapshot_download  # type: ignore

        print(f"Downloading from ModelScope: {model_id}")
        local_dir = Path(snapshot_download(model_id, cache_dir=str(cache_dir)))
        return local_dir
    except Exception as e_modelscope:
        print("ModelScope download not available or failed:")
        print("  ", repr(e_modelscope))

    # Fallback to HuggingFace Hub
    try:
        from huggingface_hub import snapshot_download as hf_snapshot_download  # type: ignore

        print(f"Downloading from HuggingFace Hub: {model_id}")
        local_dir = Path(
            hf_snapshot_download(
                repo_id=model_id,
                cache_dir=str(cache_dir),
                local_dir=str(cache_dir / "hf" / model_id.replace("/", "__")),
                local_dir_use_symlinks=False,
            )
        )
        return local_dir
    except Exception as e_hf:
        raise RuntimeError(
            "Failed to download the pretrained model.\n"
            f"- Tried ModelScope repo_id={model_id}\n"
            f"- Tried HuggingFace repo_id={model_id}\n"
            "Please set an explicit model id via environment variable: PADDLEOCR_VL_PRETRAINED_ID\n"
            "and retry."
        ) from e_hf


PRETRAINED_DIR = download_pretrained_model(MODEL_ID, CACHE_DIR)

/home/aistudio/external-libraries/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


2026-03-27 08:53:17,470 - modelscope - INFO - Target directory already exists, skipping creation.


In [5]:
# Patch the downloaded model's modeling file so that trust_remote_code uses our implementation.
# This assumes the model repo expects `modeling_paddleocr_vl.py` at repo root.

target = PRETRAINED_DIR / "modeling_paddleocr_vl.py"
backup = PRETRAINED_DIR / "modeling_paddleocr_vl.py.bak"

if target.exists() and not backup.exists():
    shutil.copy2(target, backup)

shutil.copy2(PATCH_MODELING_FILE, target)

PosixPath('/home/aistudio/openvino_notebooks/notebooks/paddleocr_vl/_cache/PaddlePaddle/PaddleOCR-VL-1___5/modeling_paddleocr_vl.py')

## Compress model
[back to top ⬆️](#Table-of-content:)

To **reduce memory usage** and, in many cases, improve inference performance for large models (especially LLMs), you can apply **weight compression**.

- **How it works**: OpenVINO weight compression stores/computes weights using **mixed INT4/INT8 quantization**, while keeping **activations in floating point**. Compared with full-model INT8 post-training quantization (PTQ), this typically helps preserve accuracy.
- **Benefits**:
  - **Significantly lower memory footprint**, enabling larger models to run on the target device
  - **Better performance** by reducing memory bandwidth pressure (especially for memory-bound LLM workloads)
- **Trade-offs**:
  - Compared with INT8, **INT4** is usually faster and smaller, but may introduce a **minor quality drop**
- **Key property**:
  - **Data-free**: no calibration dataset is required

The conversion toggles in this notebook map to the concepts above:

- **`LLM_INT4_COMPRESS = True`**: enable **INT4 weight compression** for the LLM part (smaller/faster, may slightly impact quality)
- **`LLM_INT8_COMPRESS = True`**: enable **INT8 weight compression** for the LLM part (a balanced option)
- **`VISION_INT8_QUANT = True`**: enable INT8 for the vision encoder (performance-oriented; evaluate any quality impact for your tasks)

> Tip: If you enable both `LLM_INT4_COMPRESS` and `LLM_INT8_COMPRESS`, the notebook will export **two compressed variants**, making it easy to compare speed vs. quality.



In [6]:
# Convert to OpenVINO IR (inline)

LLM_INT4_COMPRESS = False
LLM_INT8_COMPRESS = True
VISION_INT8_QUANT = False

In [7]:
# Convert to OpenVINO IR using the local exporter implementation.
# This uses the same core logic as ov_model_convert.py, but runs inline for notebook independence.

import sys

sys.path.insert(0, str(NOTEBOOK_DIR))  # ensure local imports resolve

from ov_paddleocr_vl import PaddleOCR_VL_OV

from pathlib import Path

ov_out_dir = Path(OV_OUT_DIR)

if ov_out_dir.exists():
    print("✅ OV_OUT_DIR already exists, skip conversion!")
else:
    print("Converting model to OpenVINO...")

    paddleocr_vl_ov = PaddleOCR_VL_OV(
        pretrained_model_path=str(PRETRAINED_DIR),
        ov_model_path=str(OV_OUT_DIR),
        device=device.value,
        llm_int4_compress=LLM_INT4_COMPRESS,
        llm_int8_compress=LLM_INT8_COMPRESS,
        vision_int8_quant=VISION_INT8_QUANT,
    )

    paddleocr_vl_ov.export_paddleocr_vl_to_ov()
    print("✅ Conversion complete.")

    paddleocr_vl_ov.close()
    print("✅ Resources released.")

✅ OV_OUT_DIR already exists, skip conversion!


## Run model inference

[back to top ⬆️](#Table-of-content:)

In [8]:
import openvino as ov
from ov_paddleocr_vl import OVPaddleOCRVLForCausalLM

# Parameters
ov_model_path = str(OV_OUT_DIR)
task = "ocr"
max_new_tokens = 512

llm_infer_list = []
vision_infer = []
core = ov.Core()

paddleocr_vl_model = OVPaddleOCRVLForCausalLM(
    core=core,
    ov_model_path=ov_model_path,
    device=device.value,
    llm_int4_compress=False,
    llm_int8_compress=True,
    vision_int8_quant=False,
    llm_int8_quant=True,
    llm_infer_list=llm_infer_list,
    vision_infer=vision_infer,
)

In [9]:
# Validate the end-to-end OpenVINO pipeline (inline implementation)
# NOTE: This cell expands the logic from ov_test.py::run_openvino_test() to avoid calling it directly.

import time

from PIL import Image


PROMPTS = {
    "ocr": "OCR:",
    "table": "Table Recognition:",
    "formula": "Formula Recognition:",
    "chart": "Chart Recognition:",
}

# Prepare a test image.
img = None
if img is None:
    img = NOTEBOOK_DIR / "test.png"

image_path = str(img)
image = Image.open(image_path).convert("RGB")
# image = image.resize((1200, 800), Image.Resampling.LANCZOS)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPTS[task]},
        ],
    }
]

print("\n" + "=" * 60)
print("🔄 Loading OpenVINO model...")
print("=" * 60)

# OpenVINO version print
try:
    print("OpenVINO version:\n", ov.get_version())
except Exception:
    print("OpenVINO version:\n", getattr(ov, "__version__", "unknown"))
print()

generation_config = {
    "bos_token_id": paddleocr_vl_model.tokenizer.bos_token_id,
    "eos_token_id": paddleocr_vl_model.tokenizer.eos_token_id,
    "pad_token_id": paddleocr_vl_model.tokenizer.pad_token_id,
    "max_new_tokens": int(max_new_tokens),
    "do_sample": False,
}

start = time.perf_counter()
response, history = paddleocr_vl_model.chat(messages=messages, generation_config=generation_config)
chat_time = time.perf_counter() - start


print("\n" + "=" * 60)
print(f"📄 {device.value} OpenVINO {task} result:")
print("=" * 60)
print(response)
print("=" * 60)


🔄 Loading OpenVINO model...
OpenVINO version:
 2026.0.0-20965-c6d6a13a886-releases/2026/0


📄 CPU OpenVINO ocr result:

PaddleOCR-VL-1.5 is an advanced next-generation model of PaddleOCR-VL, achieving a new state-of-the-art accuracy of 94.5% on OmniDocBench v1.5. To rigorously evaluate robustness against real-world physical distortions—including scanning artifacts, skew, warping, screen photography, and illumination—we propose the Real5-OmniDocBench benchmark. Experimental results demonstrate that this enhanced model attains SOTA performance on the newly curated benchmark. Furthermore, we extend the model's capabilities by incorporating seal recognition and text spotting tasks, while remaining a 0.9B ultra-compact VLM with high efficiency.


## Interactive demo
[back to top ⬆️](#Table-of-contents:)

请运行run.gradio.py,并在里面修改绝对路径